In [51]:
import pandas as pd

In [52]:
df=pd.read_csv("hour.csv", parse_dates=["dteday"])

In [53]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [54]:
# Features and target (RAW — no one-hot encoding yet)
categorical_cols = ["season", "mnth", "hr", "weekday", "weathersit"]
numeric_cols = ["holiday", "workingday", "temp", "hum", "windspeed"]

X = df[categorical_cols + numeric_cols]
Y = df["cnt"]


In [55]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test=train_test_split(X,Y,test_size=0.3, random_state=42)

In [56]:

# Preprocessor: OneHotEncode categorical, pass through numeric
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Pipeline: preprocessing + model
pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

In [57]:
# hyper parameter tunnig
parameter={
    'max_depth':[5,10,15,None],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,5]
}

In [58]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
dt=DecisionTreeRegressor(random_state=42)
grid=GridSearchCV(dt,param_grid=parameter, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train,Y_train)
best_model=grid.best_estimator_

In [59]:
print("DT best params:", grid.best_params_)

DT best params: {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}


In [60]:
# 2. Linear Regression baseline
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, Y_train)

LinearRegression()

In [61]:
#  Evaluate both models
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
for name , model in [("Decision Tree",best_model),("linear_Model" ,lr)]:
    Y_pred=model.predict(X_test)
    mse=mean_squared_error(Y_test,Y_pred)
    r2=r2_score(Y_test,Y_pred)
    print(f"{name} MSE: {mse:.2f}, RMSE: {np.sqrt(mse):.2f}, R2: {r2:.3f}")

Decision Tree MSE: 5861.39, RMSE: 76.56, R2: 0.814
linear_Model MSE: 20766.07, RMSE: 144.10, R2: 0.343


In [62]:
joblib.dump(grid.best_estimator_, "bike_model.pkl")
print("Model saved as bike_model.pkl")

Model saved as bike_model.pkl


ValueError: A given column is not a column of the dataframe